# COVID Task-Transfer Matrix

Fixed-adaptation transfer results for COVID-only train-task to eval-task experiments. Rows are source/pretraining tasks, columns are eval tasks after 1000 adaptation steps.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("task_matrix_adapt_1000.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

row_order = ["scratch", "nm", "cl", "fp"]
col_order = ["nm", "cl", "fp"]
labels = {"scratch": "Scratch", "nm": "NM", "cl": "CL", "fp": "FP"}

df = pd.read_csv(DATA)
df["value"] = pd.to_numeric(df["value"])
df.head()

In [ ]:
def matrix(metric):
    return (
        df[df["metric"].eq(metric)]
        .pivot(index="train_task", columns="eval_task", values="value")
        .reindex(index=row_order, columns=col_order)
    )

primary = matrix("primary")
auc = matrix("roc_auc")
accuracy = matrix("accuracy")
loss = matrix("loss")
score = matrix("score")

primary

The primary matrix mixes metrics: NM/CL use ROC-AUC, while FP uses negative MSE. The clearer plots below treat each eval task separately and compare each pretraining row against scratch.

In [ ]:
def bar_by_source(values, title, ylabel, filename, baseline=None):
    values = values.reindex(row_order)
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    colors = ["#8a8f98", "#2f6f9f", "#5a9f72", "#b45f4d"]
    ax.bar([labels[x] for x in values.index], values.values, color=colors, width=0.68)
    if baseline is not None:
        ax.axhline(baseline, color="#3a3a3a", linestyle="--", linewidth=1.2, label="Scratch")
        ax.legend(frameon=False, loc="best")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#d7d7d7", linewidth=0.8, alpha=0.7)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=200)
    return fig, ax

bar_by_source(auc["nm"], "Eval: NM after 1000 adaptation steps", "ROC-AUC", "nm_auc_by_pretrain.png", baseline=auc.loc["scratch", "nm"]);
bar_by_source(auc["cl"], "Eval: CL after 1000 adaptation steps", "ROC-AUC", "cl_auc_by_pretrain.png", baseline=auc.loc["scratch", "cl"]);
bar_by_source(loss["fp"], "Eval: FP after 1000 adaptation steps", "MSE loss (lower is better)", "fp_loss_by_pretrain.png", baseline=loss.loc["scratch", "fp"]);

In [ ]:
improvement = pd.DataFrame(index=row_order, columns=col_order, dtype=float)
for eval_task in ["nm", "cl"]:
    improvement[eval_task] = auc[eval_task] - auc.loc["scratch", eval_task]

# FP uses loss, so positive improvement means lower loss than scratch.
improvement["fp"] = loss.loc["scratch", "fp"] - loss["fp"]
improvement

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
plot_df = improvement.rename(index=labels, columns={"nm": "NM AUC", "cl": "CL AUC", "fp": "FP loss"})
plot_df.plot(kind="bar", ax=ax, width=0.78, color=["#2f6f9f", "#5a9f72", "#b45f4d"])
ax.axhline(0, color="#3a3a3a", linewidth=1.0)
ax.set_title("Improvement over scratch after fixed adaptation")
ax.set_ylabel("Delta vs scratch (FP is loss reduction)")
ax.set_xlabel("")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#d7d7d7", linewidth=0.8, alpha=0.7)
ax.legend(frameon=False, title="Eval task")
fig.tight_layout()
fig.savefig(FIG_DIR / "improvement_vs_scratch.png", dpi=200)
plot_df

In [ ]:
summary_rows = []
for eval_task in col_order:
    if eval_task in ["nm", "cl"]:
        metric_values = auc[eval_task]
        best_source = metric_values.idxmax()
        scratch_value = metric_values.loc["scratch"]
        summary_rows.append({
            "eval_task": eval_task.upper(),
            "metric": "ROC-AUC",
            "best_source": labels[best_source],
            "best_value": metric_values.loc[best_source],
            "scratch_value": scratch_value,
            "delta_vs_scratch": metric_values.loc[best_source] - scratch_value,
        })
    else:
        metric_values = loss[eval_task]
        best_source = metric_values.idxmin()
        scratch_value = metric_values.loc["scratch"]
        summary_rows.append({
            "eval_task": eval_task.upper(),
            "metric": "MSE loss",
            "best_source": labels[best_source],
            "best_value": metric_values.loc[best_source],
            "scratch_value": scratch_value,
            "delta_vs_scratch": scratch_value - metric_values.loc[best_source],
        })

summary = pd.DataFrame(summary_rows)
summary